# Milestone 2 — manual Colab real-model benchmark
Use the existing application, not a replacement pipeline. Run stages in order. **Run all is safe by default:** it resets approvals and cannot download models or run inference without fresh button clicks. No publishing, RunPod, OpenAI/Gemini API, subscription, driver install or capture.

Milestone 1: MOCK acceptance passed; smooth playback and perceived audio quality remain follow-up checks. This notebook has been structurally validated and its MOCK path tested on Windows; **Colab/CUDA/Ollama and real models have not been executed by the author**. Stop if Python <3.12. Colab resources and compute-unit availability are not guaranteed.

In [ ]:
import getpass
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
from contextlib import contextmanager
from pathlib import Path

assert sys.version_info >= (3, 12), (
    "Select a Python >=3.12 runtime; do not silently change the app."
)
# Re-running this cell invalidates all prior approvals.
APPROVALS = {}
REPOSITORY = "farkfarm/alofoke-clips-automation"
REVISION = "c1f9b16157ff72874ec554c3acca34d4ff8f8859"
PRIVATE_SOURCE = False
BASE = Path("/content") if Path("/content").exists() else Path.cwd() / "data"
WORK = BASE / ("alofoke-benchmark-" + uuid.uuid4().hex[:12])
WORK.mkdir(parents=True, exist_ok=False)
REPO = WORK / "repository"
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = sys.executable
SETUP = {}
ERRORS = []


def save_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
    )


def run(args, cwd=None, env=None, log=None):
    result = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        env=env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if log:
        (RESULTS / (log + ".stdout.txt")).write_text(result.stdout, encoding="utf-8")
        (RESULTS / (log + ".stderr.txt")).write_text(result.stderr, encoding="utf-8")
    if result.returncode:
        ERRORS.append({"stage": log or "setup", "returncode": result.returncode})
        save_json(RESULTS / "errors.json", ERRORS)
        raise RuntimeError(
            f"Command failed ({result.returncode}); inspect the stage log. No fallback was run."
        )
    return result.stdout


@contextmanager
def setup_time(name):
    start = time.perf_counter()
    try:
        yield
    except Exception as error:
        ERRORS.append({"stage": name, "error_type": type(error).__name__})
        save_json(RESULTS / "errors.json", ERRORS)
        raise
    finally:
        SETUP[name] = time.perf_counter() - start
        save_json(RESULTS / "setup-times.json", SETUP)


print("Fresh runtime-local workspace:", WORK)

## 1. Load an explicit revision
For private access set PRIVATE_SOURCE above, then enter a read-only repository token in the hidden prompt. It is passed only through a temporary child environment to Git askpass; it is never put in a URL, file, command argument, notebook output or report. No token is requested for public access. Do not paste a token into a cell. Clear runtime variables after use; do not save populated notebook outputs.

In [ ]:
import re

assert re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY)
assert REVISION and not REVISION.startswith("-") and re.fullmatch(r"[A-Za-z0-9_./-]+", REVISION)
assert not REPO.exists(), (
    "Re-run initialization for a fresh workspace; never reset an existing checkout."
)
with setup_time("source_checkout_seconds"):
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="never")
    helper = None
    try:
        if PRIVATE_SOURCE:
            token = getpass.getpass("Read-only GitHub token (hidden; do not paste into a cell): ")
            helper = WORK / "askpass.py"
            helper.write_text(
                "#!"
                + sys.executable
                + '\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["ALOFOKE_SOURCE_TOKEN"])\n'
            )
            helper.chmod(0o700)
            env.update(GIT_ASKPASS=str(helper), ALOFOKE_SOURCE_TOKEN=token)
            del token
        # Source operations deliberately do not persist stdout/stderr, which may contain auth diagnostics.
        run(["git", "init", REPO])
        run(
            ["git", "remote", "add", "origin", "https://github.com/" + REPOSITORY + ".git"],
            cwd=REPO,
        )
        run(
            ["git", "-c", "credential.helper=", "fetch", "--depth=1", "origin", REVISION],
            cwd=REPO,
            env=env,
        )
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    finally:
        env.pop("ALOFOKE_SOURCE_TOKEN", None)
        if helper:
            helper.unlink(missing_ok=True)
COMMIT = run(["git", "rev-parse", "HEAD"], cwd=REPO).strip()
save_json(
    RESULTS / "source-revision.json",
    {"repository": REPOSITORY, "requested_revision": REVISION, "commit": COMMIT},
)
print("Loaded commit:", COMMIT)

## 2. Inspect and install only baseline dependencies
No model packages or weights here. FFmpeg must have libx264/AAC/libass. If absent in Colab, deliberately install the OS ffmpeg package before proceeding; this notebook never installs drivers. Setup time includes package installation separately from processing.

In [ ]:
with setup_time("baseline_dependencies_seconds"):
    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "-e",
            str(REPO),
            "pytest>=8,<10",
            "pytest-cov>=6,<8",
            "ruff>=0.11,<1",
        ],
        log="baseline-install",
    )
os.chdir(REPO)
# Editable-install .pth files are not reprocessed in an already-running kernel.
sys.path.insert(0, str(REPO / "src"))
FFMPEG = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")
assert FFMPEG and FFPROBE, "FFmpeg/FFprobe missing: manually install ffmpeg, then rerun this stage."
from alofoke_clips.config import Settings
from alofoke_clips.diagnostics import diagnose

hardware = diagnose(Settings(data_dir=RESULTS))
save_json(RESULTS / "hardware.json", hardware)
versions = run([PYTHON, "-m", "pip", "list", "--format=json"])
(RESULTS / "baseline-packages.json").write_text(versions)
assert hardware["ffmpeg"]["ready"], hardware["ffmpeg"]
print(json.dumps(hardware, indent=2))

## 3. CPU-only MOCK smoke test
The existing generator and CLI create a fresh queue and export one fixture-driven clip. No inference SDK, model download, model server or AI API is needed. A color field and tone are not a real Spanish recording.

In [ ]:
os.environ["ALOFOKE_FFMPEG"] = FFMPEG
os.environ["ALOFOKE_FFPROBE"] = FFPROBE


def write_config(path, mode, data_dir, model_path=None):
    lines = [
        f"mode = {json.dumps(mode)}",
        f"data_dir = {json.dumps(str(data_dir))}",
        "[media]",
        f"ffmpeg = {json.dumps(FFMPEG)}",
        f"ffprobe = {json.dumps(FFPROBE)}",
        "threads = 2",
        'preset = "veryfast"',
        "[selection]",
        "max_clips = 3",
        "[gpu]",
        "allow_paid = false",
        "monthly_budget_usd = 0",
    ]
    if mode == "local":
        lines += [
            "[transcription]",
            f"model_path = {json.dumps(str(model_path))}",
            'device = "cuda"',
            'compute_type = "int8_float16"',
            "cpu_fallback = false",
            "[analysis]",
            'backend = "ollama"',
            'base_url = "http://127.0.0.1:11434"',
            f"model = {json.dumps(QWEN_ID)}",
            "unload_after_analysis = true",
        ]
    Path(path).write_text("\n".join(lines) + "\n")


MOCK_CONFIG = WORK / "mock.toml"
write_config(MOCK_CONFIG, "mock", WORK / "mock-output")
FIXTURE = WORK / "synthetic"
run(
    [
        PYTHON,
        "scripts/make_synthetic.py",
        "--config",
        MOCK_CONFIG,
        "--output",
        FIXTURE,
        "--seconds",
        "22",
    ],
    cwd=REPO,
    log="mock-generate",
)
mock = json.loads(
    run(
        [
            PYTHON,
            "-m",
            "alofoke_clips",
            "--config",
            MOCK_CONFIG,
            "process",
            FIXTURE / "source.mp4",
            "--authorized-local",
            "--mode",
            "mock",
            "--mock-transcript",
            FIXTURE / "transcript.json",
            "--mock-analysis",
            FIXTURE / "analysis.json",
        ],
        cwd=REPO,
        log="mock-process",
    )
)
assert mock["status"] == "completed" and not mock["errors"] and len(mock["clips"]) == 1
assert mock["transcription_metrics"]["model_loaded"] is False
assert mock["analysis_metrics"]["model_loaded"] is False
save_json(RESULTS / "mock-report.json", mock)
print("CPU-compatible MOCK passed; output:", mock["clips"][0])

## 4. Review the proposal — no downloads yet
Whisper: **Systran/faster-whisper-small**, multilingual, revision **536b0662742c02347bc0e980a01041f333bce120**, approximately **486 MB**. Qwen: **qwen3:4b**, Q4_K_M, approximately **2.5 GB** (weights only; runtime/context memory is additional). Ollama binary and inference dependencies require additional downloads. Sources: [Whisper files](https://huggingface.co/Systran/faster-whisper-small/tree/main), [Qwen listing](https://ollama.com/library/qwen3:4b).

Select an authorized ten-minute recording in stage 5, then click the download approval button. A second button is required immediately before real inference. Full Run all resets approvals; missing approval fails closed. Clicks approve this workspace/proposal only, are consumed once, and never persist in notebook files.

In [ ]:
WHISPER_ID = "Systran/faster-whisper-small"
WHISPER_REVISION = "536b0662742c02347bc0e980a01041f333bce120"
QWEN_ID = "qwen3:4b"
PROPOSAL = {
    "whisper_id": WHISPER_ID,
    "whisper_revision": WHISPER_REVISION,
    "whisper_approx_MB": 486,
    "qwen_id": QWEN_ID,
    "qwen_quantization": "Q4_K_M",
    "qwen_approx_GB": 2.5,
    "device": "cuda",
    "compute_type": "int8_float16",
    "cpu_fallback": False,
    "max_clips": 3,
    "encoding": "CPU libx264",
    "server": "127.0.0.1:11434",
    "publishing": "disabled",
}
print(json.dumps(PROPOSAL, indent=2))
import ipywidgets as widgets
from IPython.display import display


def approval_button(kind, context):
    button = widgets.Button(description="Approve " + kind, button_style="warning")

    def clicked(_):
        APPROVALS[kind] = context()
        button.disabled = True
        button.description = "Approved once"

    button.on_click(clicked)
    display(button)


def proposal_key():
    return hashlib.sha256(json.dumps(PROPOSAL, sort_keys=True).encode()).hexdigest() + str(WORK)


approval_button("downloads", proposal_key)

## 5. Explicit source selection and permission
Mount Drive only if you choose to. Set a specific existing recording path below (or use Colab's Files panel upload, then select its path). No automatic Drive scan or source replacement. Copy to runtime-local storage before processing. The `benchmark` command expects 600 ±30 seconds; extract your permitted sample separately if necessary.

In [ ]:
# Optional, manual mount cell: uncomment only when desired.
# from google.colab import drive
# drive.mount('/content/drive')
SELECTED_SOURCE = ""  # e.g. /content/drive/MyDrive/authorized/sample-10min.mp4
assert SELECTED_SOURCE, "Select one permitted recording explicitly."
source = Path(SELECTED_SOURCE).expanduser().resolve()
assert source.is_file() and source.suffix.lower() in {
    ".mp4",
    ".mkv",
    ".mov",
    ".webm",
    ".ts",
    ".avi",
    ".m4v",
}
LOCAL_SOURCE = WORK / "input" / ("recording" + source.suffix.lower())
LOCAL_SOURCE.parent.mkdir(exist_ok=True)
assert not LOCAL_SOURCE.exists(), "Use a fresh workspace; do not overwrite inputs."
with setup_time("source_copy_seconds"):
    shutil.copyfile(source, LOCAL_SOURCE)
from alofoke_clips.config import MediaConfig
from alofoke_clips.media import MediaTools

info = MediaTools(MediaConfig(ffmpeg=FFMPEG, ffprobe=FFPROBE)).validate(LOCAL_SOURCE)
assert abs(info.duration - 600) <= 30, "Select a 600 +/-30 second permitted sample."
SOURCE_SHA = hashlib.file_digest(LOCAL_SOURCE.open("rb"), "sha256").hexdigest()
SOURCE_ORIGINAL_SHA = hashlib.file_digest(source.open("rb"), "sha256").hexdigest()
assert SOURCE_SHA == SOURCE_ORIGINAL_SHA
print("Local copy duration:", info.duration, "seconds; source has not been modified.")

## 6a. Approved runtime/model setup
Requires the fresh **downloads** button click, valid local source, and an assigned NVIDIA GPU. No driver installation or paid runtime purchase. If Ollama is absent, install its Linux binary following [official instructions](https://docs.ollama.com/linux) manually, then rerender/reapprove the button. The notebook starts only the existing binary, bound to loopback, and refuses an occupied port rather than reusing an unknown server. It pulls models only after approval. CUDA/cuDNN compatibility failures remain errors to investigate, never mock substitutions.

In [ ]:
assert LOCAL_SOURCE.is_file()
assert hardware["nvidia"]["available"], (
    "No NVIDIA GPU assigned; MOCK works, real CUDA benchmark blocked."
)
OLLAMA = shutil.which("ollama")
assert OLLAMA, (
    "Manually install Ollama Linux binary per docs; no installer is executed by this notebook."
)
assert APPROVALS.pop("downloads", None) == proposal_key(), (
    "Click Approve downloads after reviewing proposal."
)
with setup_time("inference_dependencies_seconds"):
    run(
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "-e",
            str(REPO) + "[transcribe]",
            "huggingface-hub>=0.30,<2",
        ],
        log="inference-install",
    )
from huggingface_hub import snapshot_download

with setup_time("whisper_download_seconds"):
    WHISPER_PATH = Path(
        snapshot_download(
            WHISPER_ID,
            revision=WHISPER_REVISION,
            local_dir=WORK / "models/whisper",
            allow_patterns=["model.bin", "config.json", "tokenizer.json", "vocabulary.*"],
        )
    )
with setup_time("ollama_server_startup_seconds"):
    import socket

    with socket.socket() as sock:
        assert sock.connect_ex(("127.0.0.1", 11434)) != 0, (
            "Port11434 already occupied; stop unknown server explicitly."
        )
    SERVER_LOG = (WORK / "ollama-server.log").open("w")
    server_env = dict(
        os.environ,
        OLLAMA_HOST="127.0.0.1:11434",
        OLLAMA_MODELS=str(WORK / "models/ollama"),
        OLLAMA_NO_CLOUD="1",
    )
    SERVER = subprocess.Popen(
        [OLLAMA, "serve"], env=server_env, stdout=SERVER_LOG, stderr=subprocess.STDOUT
    )
    from alofoke_clips.providers.http import JsonHTTP

    HTTP = JsonHTTP()
    for _attempt in range(60):
        try:
            HTTP.request("http://127.0.0.1:11434/api/version", None, 2)
            break
        except Exception:
            if SERVER.poll() is not None:
                raise RuntimeError("Ollama exited; inspect runtime-local server log.") from None
            time.sleep(1)
    else:
        raise RuntimeError("Ollama startup timeout; stop server in cleanup cell.")
with setup_time("qwen_download_seconds"):
    pull = HTTP.request(
        "http://127.0.0.1:11434/api/pull", {"model": QWEN_ID, "stream": False}, 7200
    )
    assert pull.get("status") == "success"
tags = HTTP.request("http://127.0.0.1:11434/api/tags", None, 30)
QWEN_MANIFEST = next(m for m in tags["models"] if m["name"] == QWEN_ID)
assert QWEN_MANIFEST["details"]["quantization_level"] == "Q4_K_M"
MODEL_VERSIONS = {
    "proposal": PROPOSAL,
    "whisper_resolved_revision": WHISPER_REVISION,
    "qwen_manifest": QWEN_MANIFEST,
    "ollama": HTTP.request("http://127.0.0.1:11434/api/version", None, 10),
    "packages": json.loads(run([PYTHON, "-m", "pip", "list", "--format=json"])),
}
save_json(RESULTS / "model-versions.json", MODEL_VERSIONS)
print("Downloaded exact Qwen digest:", QWEN_MANIFEST["digest"])
print("Review model-versions.json before approving real inference.")

## 6b. Approve one real run and record compute units manually
Obtain your own current Colab balance from the UI. Leave unknown values as None. No balance is read or estimated automatically. The approval is tied to the selected local source hash, proposal and downloaded Qwen digest. Rendering uses CPU libx264; no publication occurs.

In [ ]:
COMPUTE_UNITS_BEFORE = None  # enter a number from your account UI, or leave unknown
COMPUTE_UNITS_AFTER = None
assert MODEL_VERSIONS and LOCAL_SOURCE.is_file()


def real_key():
    return proposal_key() + SOURCE_SHA + QWEN_MANIFEST["digest"]


approval_button("real inference", real_key)

## 6c–7. Existing LOCAL benchmark and up to three clips
This invokes `python -m alofoke_clips ... benchmark`, including its queue, provider factory, quality/abstention checks, subtitles, CPU rendering and metadata. Only the existing Whisper model constructor is instrumented to measure load time; no replacement transcriber/analyzer is introduced. Whisper is released before Qwen starts, and Qwen is unloaded after analysis. A failed real run stays failed. Zero selected clips is valid.

In [ ]:
assert APPROVALS.pop("real inference", None) == real_key(), (
    "Click Approve real inference for this run."
)
assert hashlib.file_digest(LOCAL_SOURCE.open("rb"), "sha256").hexdigest() == SOURCE_SHA
assert SERVER.poll() is None
REAL_CONFIG = WORK / "real.toml"
write_config(REAL_CONFIG, "local", RESULTS / "pipeline", WHISPER_PATH)
# Child instrumentation measures only construction of the existing provider's underlying model.
launcher = WORK / "instrument_cli.py"
launcher.write_text("""import json,sys,time,runpy
from pathlib import Path
import faster_whisper
original=faster_whisper.WhisperModel
loads=[]
class TimedWhisper(original):
    def __init__(self,*a,**kw):
        t=time.perf_counter()
        try: super().__init__(*a,**kw)
        finally:
            loads.append(time.perf_counter()-t)
            Path(sys.argv[1]).write_text(json.dumps(loads))
faster_whisper.WhisperModel=TimedWhisper
load_file=sys.argv.pop(1)
sys.argv[0]='alofoke_clips'
runpy.run_module('alofoke_clips',run_name='__main__')
""")
# Fix output path capture before CLI arguments are shifted.
text = launcher.read_text().replace("Path(sys.argv[1]).write_text", "Path(load_file).write_text")
launcher.write_text(text)
REAL_START = time.perf_counter()
try:
    raw = run(
        [
            PYTHON,
            launcher,
            RESULTS / "whisper-load-seconds.json",
            "--config",
            REAL_CONFIG,
            "benchmark",
            LOCAL_SOURCE,
            "--authorized-local",
            "--mode",
            "local",
        ],
        cwd=REPO,
        log="real-benchmark",
    )
    REAL_REPORT = json.loads(raw)
except Exception:
    reports = list((RESULTS / "pipeline/jobs").glob("*/report.json"))
    REAL_REPORT = (
        json.loads(reports[-1].read_text()) if reports else {"status": "failed", "errors": ERRORS}
    )
finally:
    REAL_WALL = time.perf_counter() - REAL_START
    save_json(RESULTS / "real-report.json", REAL_REPORT)
assert REAL_REPORT["status"] == "completed", (
    "Real inference failed. Export errors; do not replace with MOCK."
)
assert REAL_REPORT["mode"] == "local" and REAL_REPORT["publication_status"] == "disabled"
assert len(REAL_REPORT["clips"]) <= 3
assert REAL_REPORT["transcription_metrics"]["provider"] == "faster_whisper"
assert REAL_REPORT["transcription_metrics"]["device"] == "cuda"
assert REAL_REPORT["analysis_metrics"]["provider"] == "ollama"
print("Real pipeline completed; selected clips:", len(REAL_REPORT["clips"]))

## 8. Final report, manual unit readings and export
Enter an after-balance in the following cell **after** the real run. The difference is not an inference cost estimate: other sessions or runtime idle time may consume units. Missing readings remain null. Read all errors before sharing. Archives include transcripts/captions/derived video (potentially sensitive) but exclude tokens, source recording, model weights, repository, environment and notebook output.

In [ ]:
# Edit these manually, then run this cell after completion or failure.
COMPUTE_UNITS_AFTER = None
REAL_REPORT = globals().get("REAL_REPORT", {"status": "not_run", "errors": ERRORS})
REAL_WALL = globals().get("REAL_WALL")
MODEL_VERSIONS = globals().get("MODEL_VERSIONS", {"proposal": PROPOSAL})
COMPUTE_UNITS_BEFORE = globals().get("COMPUTE_UNITS_BEFORE")
loads = (
    json.loads((RESULTS / "whisper-load-seconds.json").read_text())
    if (RESULTS / "whisper-load-seconds.json").exists()
    else []
)
calls = REAL_REPORT.get("analysis_metrics", {}).get("calls", [])
qwen_load = sum(c.get("load_duration", 0) for c in calls) / 1e9
stages = REAL_REPORT.get("stage_seconds", {})
before = COMPUTE_UNITS_BEFORE
after = COMPUTE_UNITS_AFTER
assert all(
    v is None or (isinstance(v, (int, float)) and not isinstance(v, bool) and v >= 0)
    for v in (before, after)
)
summary = {
    "commit": COMMIT,
    "source_duration_seconds": info.duration,
    "source_sha256": SOURCE_SHA,
    "setup_seconds": SETUP,
    "model_versions": MODEL_VERSIONS,
    "whisper_model_load_seconds": loads,
    "qwen_reported_load_seconds": qwen_load if calls else None,
    "transcription_stage_seconds_including_load": stages.get("transcription"),
    "transcription_seconds_excluding_measured_constructor": max(
        0, stages["transcription"] - sum(loads)
    )
    if "transcription" in stages and loads
    else None,
    "analysis_stage_seconds_including_load": stages.get("analysis"),
    "analysis_seconds_excluding_reported_load": max(0, stages["analysis"] - qwen_load)
    if "analysis" in stages and calls
    else None,
    "rendering_seconds": stages.get("clip_rendering", 0),
    "pipeline_total_seconds": REAL_REPORT.get("total_seconds"),
    "cli_wall_seconds": REAL_WALL,
    "peak_whole_device_gpu_used_mib": REAL_REPORT.get("peak_device_gpu_used_mib"),
    "gpu_samples": REAL_REPORT.get("gpu_memory_samples"),
    "analysis_server_memory": REAL_REPORT.get("analysis_server_memory"),
    "errors": REAL_REPORT.get("errors", ERRORS),
    "status": REAL_REPORT.get("status"),
    "compute_units_before_manual": before,
    "compute_units_after_manual": after,
    "compute_units_difference_manual": before - after
    if before is not None and after is not None
    else None,
    "cost_usd": None,
    "a100_performance": None,
    "limitations": [
        "1Hz whole-device NVIDIA sampling includes other applications and can miss peaks.",
        "Python RSS excludes the separate Ollama server; Ollama snapshot is separate.",
        "Constructor timing excludes possible lazy loading during decoding. Qwen load timing is server-reported.",
        "Derived durations are estimates; stage totals remain authoritative. No throughput or hardware recommendation yet.",
        "Smooth playback, perceived audio and Spanish editorial quality require manual review.",
    ],
}
assert hashlib.file_digest(source.open("rb"), "sha256").hexdigest() == SOURCE_ORIGINAL_SHA
save_json(RESULTS / "benchmark-summary.json", summary)
# Export an explicit whitelist; never source, models, .env or virtual environments.
EXPORT = WORK / "export"
assert not EXPORT.exists(), "Export once to a fresh destination."
EXPORT.mkdir()
for name in [
    "benchmark-summary.json",
    "setup-times.json",
    "source-revision.json",
    "model-versions.json",
    "hardware.json",
    "real-report.json",
    "errors.json",
    "whisper-load-seconds.json",
]:
    if (RESULTS / name).is_file():
        shutil.copyfile(RESULTS / name, EXPORT / name)
if (RESULTS / "pipeline/jobs").is_dir():
    shutil.copytree(
        RESULTS / "pipeline/jobs",
        EXPORT / "jobs",
        ignore=shutil.ignore_patterns("*.wav", "*.tmp", "*.partial.mp4"),
    )
ARCHIVE = Path(shutil.make_archive(str(WORK / "alofoke-results"), "zip", EXPORT))
EXPORT_DRIVE_FOLDER = ""  # explicitly select an existing mounted Drive folder, or leave blank
if EXPORT_DRIVE_FOLDER:
    destination = Path(EXPORT_DRIVE_FOLDER).resolve()
    assert destination.is_dir() and str(destination).startswith("/content/drive/"), (
        "Select a mounted Drive folder."
    )
    target = destination / (WORK.name + "-results.zip")
    with target.open("xb") as out, ARCHIVE.open("rb") as src:
        shutil.copyfileobj(src, out)
    print("New archive copied to selected Drive folder:", target)
else:
    from google.colab import files

    files.download(str(ARCHIVE))
print("Inspect exported benchmark-summary.json. No costs or compute-unit readings were invented.")

## Cleanup (also run after failure)
Stop only the Ollama process created by this notebook. No files or source recording are deleted. Export evidence first. Do not attach Colab to a persistent production queue; its runtime is temporary. [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
if "SERVER" in globals() and SERVER.poll() is None:
    SERVER.terminate()
    try:
        SERVER.wait(timeout=15)
    except subprocess.TimeoutExpired:
        SERVER.kill()
        SERVER.wait(timeout=5)
if "SERVER_LOG" in globals():
    SERVER_LOG.close()
APPROVALS.clear()
print("Notebook-owned model server stopped; source recording preserved.")